# Iterative Imputer

## Advantages
- Uses relationships between features to predict missing values.
- More accurate than simple mean/median imputation.

## Disadvantages
- Computationally slower than simple imputation.
- Can overfit if dataset is very small.

## Why We Use Iterative Imputer
Iterative Imputer estimates missing values by modeling each feature as a function of other features.  
It is useful when features are correlated and missing values are not completely random.

## **Import Libraries**

In [106]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression

## **Load and Prepare Dataset**

In [107]:
df = np.round(pd.read_csv("50_Startups.csv")[["R&D Spend", "Administration", "Marketing Spend", "Profit"]] / 10000)

# For simplicity, select small sample
np.random.seed(9)
df = df.sample(5)

# Remove target column (Profit)
df = df.iloc[:, 0:-1]

df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,4.0,5.0,20.0
2,15.0,10.0,41.0
14,12.0,16.0,26.0
44,2.0,15.0,3.0


## **Introduce Missing Values**

In [108]:
df.iloc[1, 0] = np.nan
df.iloc[3, 1] = np.nan
df.iloc[-1, -1] = np.nan

df

,R&D Spend,Administration,Marketing Spend
21,8.0,15.0,30.0
37,NaN,5.0,20.0
2,15.0,10.0,41.0
14,12.0,NaN,26.0
44,2.0,15.0,NaN


## **Apply Iterative Imputation**

In [109]:
# Create 0th iteration (Mean Imputation)
df_mean = df.fillna(df.mean())

# Copy for iterative process
df_imputed = df_mean.copy()

In [110]:
df_previous = df_imputed.copy()

for iteration in range(30):
    
    df_old = df_imputed.copy()
    
    for column in df.columns:
        
        missing_mask = df[column].isnull()
        
        if missing_mask.sum() == 0:
            continue
        
        X_train = df_imputed.loc[~missing_mask].drop(columns=[column])
        y_train = df_imputed.loc[~missing_mask, column]
        X_missing = df_imputed.loc[missing_mask].drop(columns=[column])
        
        model = LinearRegression()
        model.fit(X_train, y_train)
        
        predicted_values = model.predict(X_missing)
        df_imputed.loc[missing_mask, column] = predicted_values
    
    # Check change between iterations
    change = abs(df_imputed - df_old).sum().sum()
    print(f"Iteration {iteration+1}, Change: {change}")

Iteration 1, Change: 16.429815483114183
Iteration 2, Change: 8.642825855506203
Iteration 3, Change: 28.053061150134663
Iteration 4, Change: 8.64082214061818
Iteration 5, Change: 0.38923835741887913
Iteration 6, Change: 0.01791586858677796
Iteration 7, Change: 0.000837107110770674
Iteration 8, Change: 3.9088185751268156e-05
Iteration 9, Change: 1.8252528963813575e-06
Iteration 10, Change: 8.52314308019686e-08
Iteration 11, Change: 3.979939933174137e-09
Iteration 12, Change: 1.85730542057172e-10
Iteration 13, Change: 8.72724115197343e-12
Iteration 14, Change: 4.742872761198669e-13
Iteration 15, Change: 1.7763568394002505e-14
Iteration 16, Change: 6.039613253960852e-14
Iteration 17, Change: 1.0658141036401503e-14
Iteration 18, Change: 7.105427357601002e-14
Iteration 19, Change: 6.039613253960852e-14
Iteration 20, Change: 1.0658141036401503e-14
Iteration 21, Change: 7.105427357601002e-14
Iteration 22, Change: 6.039613253960852e-14
Iteration 23, Change: 1.0658141036401503e-14
Iteration 24, 

In [111]:
difference = df_imputed - df_mean
difference

,R&D Spend,Administration,Marketing Spend
21,0.000000,0.000000,0.000000
37,17.468364,0.000000,0.000000
2,0.000000,0.000000,0.000000
14,0.000000,1.772368,0.000000
44,0.000000,0.000000,41.442067


## **Manuaally USED Iteration**

In [112]:
# Impute all missing values with mean of respective col

df0 = pd.DataFrame()

df0['R&D Spend'] = df['R&D Spend'].fillna(df['R&D Spend'].mean())
df0['Administration'] = df['Administration'].fillna(df['Administration'].mean())
df0['Marketing Spend'] = df['Marketing Spend'].fillna(df['Marketing Spend'].mean())

In [113]:
# 0th Iteration
df0

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,9.25,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


In [114]:
# Remove the col1 imputed value
df1 = df0.copy()

df1.iloc[1,0] = np.nan

df1

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.25,26.00
44,2.0,15.00,29.25


In [115]:
# Use first 3 rows to build a model and use the last for prediction

X = df1.iloc[[0,2,3,4],1:3]
X

,Administration,Marketing Spend
21,15.00,30.00
2,10.00,41.00
14,11.25,26.00
44,15.00,29.25


In [116]:
y = df1.iloc[[0,2,3,4],0]
y

21     8.0
2     15.0
14    12.0
44     2.0
Name: R&D Spend, dtype: float64

In [117]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[1,1:].values.reshape(1,2))

c:\Users\Awais\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([23.14158651])

In [118]:
df1.iloc[1,0] = 23.14

In [119]:
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.25,26.00
44,2.00,15.00,29.25


In [120]:
# Remove the col2 imputed value

df1.iloc[3,1] = np.nan

df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.0,30.00
37,23.14,5.0,20.00
2,15.00,10.0,41.00
14,12.00,NaN,26.00
44,2.00,15.0,29.25


In [121]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[[0,1,2,4],[0,2]]
X

,R&D Spend,Marketing Spend
21,8.00,30.00
37,23.14,20.00
2,15.00,41.00
44,2.00,29.25


In [122]:
y = df1.iloc[[0,1,2,4],1]
y

21    15.0
37     5.0
2     10.0
44    15.0
Name: Administration, dtype: float64

In [123]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[3,[0,2]].values.reshape(1,2))

c:\Users\Awais\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([11.06331285])

In [124]:
df1.iloc[3,1] = 11.06

In [125]:
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,29.25


In [126]:
# Remove the col3 imputed value
df1.iloc[4,-1] = np.nan

df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.0
37,23.14,5.00,20.0
2,15.00,10.00,41.0
14,12.00,11.06,26.0
44,2.00,15.00,NaN


In [127]:
# Use last 3 rows to build a model and use the first for prediction
X = df1.iloc[0:4,0:2]
X

,R&D Spend,Administration
21,8.00,15.00
37,23.14,5.00
2,15.00,10.00
14,12.00,11.06


In [128]:
y = df1.iloc[0:4,-1]
y

21    30.0
37    20.0
2     41.0
14    26.0
Name: Marketing Spend, dtype: float64

In [129]:
lr = LinearRegression()
lr.fit(X,y)
lr.predict(df1.iloc[4,0:2].values.reshape(1,2))

c:\Users\Awais\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([31.56351448])

In [130]:
df1.iloc[4,-1] = 31.56

In [131]:
# After 1st Iteration
df1

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.14,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.06,26.00
44,2.00,15.00,31.56


In [132]:
# Subtract 0th iteration from 1st iteration

df1 - df0

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,13.89,0.00,0.00
2,0.00,0.00,0.00
14,0.00,-0.19,0.00
44,0.00,0.00,2.31


In [133]:
df2 = df1.copy()

df2.iloc[1,0] = np.nan

df2

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.06,26.00
44,2.0,15.00,31.56


In [134]:
X = df2.iloc[[0,2,3,4],1:3]
y = df2.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[1,1:].values.reshape(1,2))

c:\Users\Awais\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([23.78627207])

In [135]:
df2.iloc[1,0] = 23.78

In [136]:
df2.iloc[3,1] = np.nan
X = df2.iloc[[0,1,2,4],[0,2]]
y = df2.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[3,[0,2]].values.reshape(1,2))

c:\Users\Awais\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([11.22020174])

In [137]:
df2.iloc[3,1] = 11.22

In [138]:
df2.iloc[4,-1] = np.nan

X = df2.iloc[0:4,0:2]
y = df2.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df2.iloc[4,0:2].values.reshape(1,2))

c:\Users\Awais\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([38.87979054])

In [139]:
df2.iloc[4,-1] = 31.56

In [140]:
df2

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,23.78,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.22,26.00
44,2.00,15.00,31.56


In [141]:
df2 - df1

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.0
37,0.64,0.00,0.0
2,0.00,0.00,0.0
14,0.00,0.16,0.0
44,0.00,0.00,0.0


In [142]:
df3 = df2.copy()

df3.iloc[1,0] = np.nan

df3

,R&D Spend,Administration,Marketing Spend
21,8.0,15.00,30.00
37,NaN,5.00,20.00
2,15.0,10.00,41.00
14,12.0,11.22,26.00
44,2.0,15.00,31.56


In [143]:
X = df3.iloc[[0,2,3,4],1:3]
y = df3.iloc[[0,2,3,4],0]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[1,1:].values.reshape(1,2))

c:\Users\Awais\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([24.57698058])

In [144]:
df3.iloc[1,0] = 24.57

In [145]:
df3.iloc[3,1] = np.nan
X = df3.iloc[[0,1,2,4],[0,2]]
y = df3.iloc[[0,1,2,4],1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[3,[0,2]].values.reshape(1,2))

c:\Users\Awais\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([11.37282844])

In [146]:
df3.iloc[3,1] = 11.37

In [147]:
df3.iloc[4,-1] = np.nan

X = df3.iloc[0:4,0:2]
y = df3.iloc[0:4,-1]

lr = LinearRegression()
lr.fit(X,y)
lr.predict(df3.iloc[4,0:2].values.reshape(1,2))

c:\Users\Awais\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


array([45.53976417])

In [148]:
df3.iloc[4,-1] = 45.53

In [149]:
df2.iloc[3,1] = 11.22

In [150]:
df3

,R&D Spend,Administration,Marketing Spend
21,8.00,15.00,30.00
37,24.57,5.00,20.00
2,15.00,10.00,41.00
14,12.00,11.37,26.00
44,2.00,15.00,45.53


In [151]:
df3 - df2

,R&D Spend,Administration,Marketing Spend
21,0.00,0.00,0.00
37,0.79,0.00,0.00
2,0.00,0.00,0.00
14,0.00,0.15,0.00
44,0.00,0.00,13.97
